In [2]:
import json
import pandas as pd

# load

data = []

for i in range(4):
    file_path = f"dblp-ref/dblp-ref-{i}.json"
    print(f"Loading {file_path}...")
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))

df = pd.DataFrame(data)

print(f"Loaded: {df.shape[0]}")

Loading dblp-ref/dblp-ref-0.json...
Loading dblp-ref/dblp-ref-1.json...
Loading dblp-ref/dblp-ref-2.json...
Loading dblp-ref/dblp-ref-3.json...
Loaded: 3079007


In [ ]:
# clean + filter noise

print(f"Initial: {df.shape[0]}") # size

df = df.dropna(subset=['venue', 'title', 'abstract']) # drop missing
df = df[(df['venue'] != '') & (df['title'] != '') & (df['abstract'] != '')]

df = df[~df['venue'].str.contains('arXiv', case=False, na=False)] # drop arXiv venues

venue_counts = df['venue'].value_counts() # well known venues wih => 1000 papers
valid_venues = venue_counts[venue_counts >= 1000].index
df = df[df['venue'].isin(valid_venues)]

df['combined_text'] = df['title'] + " " + df['abstract']

print(f"After cleaning: {df.shape}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# TF-IDF
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['combined_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# PCA
n = 100 # condensed components
pca = TruncatedSVD(n_components=n, random_state=3)
reduced = pca.fit_transform(tfidf_matrix)

print(f"Reduced features shape: {reduced.shape}")